## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-09-05-45-34 +0000,nyt,"Targeted by U.S., Armed Group Strikes Back in ...",https://www.nytimes.com/2026/11/08/world/ameri...
1,2026-11-09-05-35-50 +0000,nypost,Rowdy teens cause carnival ride to swing out o...,https://nypost.com/2026/11/09/us-news/rowdy-te...
2,2026-11-09-05-31-54 +0000,latimes,More than a dozen runners fall ill at Santa Ba...,https://www.latimes.com/california/story/2026-...
3,2026-11-09-05-15-10 +0000,nypost,Trump says he ‘cannot’ forgive his enemies eve...,https://nypost.com/2026/11/09/us-news/trump-sa...
4,2026-11-09-05-01-06 +0000,nyt,"Germans Miss Merkel, Their Old Chancellor, Tho...",https://www.nytimes.com/2026/11/09/world/europ...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2678/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
22,trump,34
392,white,16
52,new,15
251,democrats,13
111,nvidia,12


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
55,2026-11-09-00-30-00 +0000,wsj,President Trump said Sunday that he was joking...,https://www.wsj.com/politics/policy/trump-says...,147
167,2026-11-08-15-21-05 +0000,nypost,House leader Hakeem Jeffries says Dems will re...,https://nypost.com/2026/11/08/us-news/hakeem-j...,115
162,2026-11-08-15-57-44 +0000,chitri,What Donald Trump’s new White House chief of s...,https://www.chicagotribune.com/2026/11/08/step...,108
54,2026-11-09-00-31-25 +0000,wapo,Trump says he was ‘only kidding’ about offerin...,https://www.washingtonpost.com/politics/2026/1...,103
71,2026-11-08-23-37-53 +0000,nypost,The other Dana White responds to Trump’s surpr...,https://nypost.com/2026/11/08/us-news/other-da...,103


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
55,147,2026-11-09-00-30-00 +0000,wsj,President Trump said Sunday that he was joking...,https://www.wsj.com/politics/policy/trump-says...
167,61,2026-11-08-15-21-05 +0000,nypost,House leader Hakeem Jeffries says Dems will re...,https://nypost.com/2026/11/08/us-news/hakeem-j...
31,60,2026-11-09-02-00-00 +0000,wsj,Voters fed up with high prices gave Democrats ...,https://www.wsj.com/politics/elections/democra...
107,56,2026-11-08-20-52-22 +0000,nypost,Nvidia CEO faces probe over affair with allege...,https://nypost.com/2026/11/08/tech/jensen-huan...
60,40,2026-11-09-00-23-00 +0000,wsj,Two former senior Trump advisers have been sen...,https://www.wsj.com/politics/policy/former-tru...
178,35,2026-11-08-14-32-22 +0000,nypost,Hero Polish cop who took down Krakow mass shoo...,https://nypost.com/2026/11/08/world-news/hero-...
21,30,2026-11-09-02-54-52 +0000,startribune,"Wolves run win streak to four games, pulling a...",https://www.startribune.com/minnesota-timberwo...
98,29,2026-11-08-21-37-36 +0000,bbc,Aramco Tower hit in heart of Saudi capital as ...,https://www.bbc.co.uk/news/articles/c6ly3mjx95...
38,28,2026-11-09-01-30-37 +0000,nypost,California dad stabs wife to death in front of...,https://nypost.com/2026/11/08/us-news/californ...
151,28,2026-11-08-17-00-00 +0000,wsj,"Justice Department lawyer Stanley Woodward, a ...",https://www.wsj.com/us-news/law/how-a-former-d...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
